## 1. Download catalog and human response CSVs

In [2]:
import json
from pathlib import Path

import pandas as pd
from huggingface_hub import snapshot_download
from IPython.display import display

REPO_ID = "LLM-Digital-Twin/Twin-2K-500"
LOCAL_DIR = Path("hf")
CSV_DIR = LOCAL_DIR / "question_catalog_and_human_response_csv"

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR,
    allow_patterns=["question_catalog_and_human_response_csv/*"],
)
sorted(p.name for p in CSV_DIR.iterdir())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

['generate_catalog_and_csvs.py',
 'question_catalog.json',
 'question_catalog_README.md',
 'wave1_3_response.csv',
 'wave1_3_response_label.csv',
 'wave4_response.csv',
 'wave4_response_label.csv']

## 2. Question catalog

In [4]:
import textwrap

catalog_df = pd.DataFrame(json.loads((CSV_DIR / "question_catalog.json").read_text()))


def extract_selector(row):
    if row["QuestionType"] == "DB":
        if pd.notna(row["Settings"]):
            raise ValueError(f"{row['QuestionID']}: DB row has unexpected Settings: {row['Settings']}")
        return ""
    return row["Settings"]["Selector"]


def print_catalog_entry(q, keys, width=90):
    for k in keys:
        val = q[k]
        if k == "QuestionText":
            wrapped = "\n".join(
                textwrap.fill(line, width=width, initial_indent="    ", subsequent_indent="    ")
                for line in str(val).splitlines() or [""]
            )
            print(f"  {k}:\n{wrapped}")
        else:
            print(f"  {k}: {val}")


catalog_df["Selector"] = catalog_df.apply(extract_selector, axis=1)
catalog_df["n_csv_cols"] = catalog_df["csv_columns"].map(len)

print(f"{len(catalog_df)} catalog entries ({catalog_df['QuestionID'].nunique()} unique QuestionIDs) -> {catalog_df['n_csv_cols'].sum()} CSV columns")
display(
    catalog_df.groupby(["QuestionType", "Selector"], as_index=False)
    .agg(n_catalog_entries=("QuestionID", "count"), n_csv_cols=("n_csv_cols", "sum"))
)
catalog_df.head(2)

256 catalog entries (246 unique QuestionIDs) -> 760 CSV columns


,QuestionType,Selector,n_catalog_entries,n_csv_cols
0,DB,,14,0
1,MC,MAHR,1,4
2,MC,MAVR,20,80
3,MC,SAHR,25,25
4,MC,SAVR,129,129
5,Matrix,Bipolar,13,158
6,Matrix,Likert,23,290
7,Slider,HSLIDER,3,12
8,TE,FORM,4,38
9,TE,ML,3,3


,QuestionID,QuestionText,QuestionType,Options,Settings,BlockName,source,csv_columns,is_descriptive,Rows,Columns,RowsID,Statements,Range,StatementsID,Selector,n_csv_cols
0,QID11,Which part of the United States do you current...,MC,"[Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME...","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID11],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1
1,QID12,What is the sex that you were assigned at birth?,MC,"[Male, Female]","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID12],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1


## 3. Human responses: waves 1–3 and wave 4

In [5]:
w13_df = pd.read_csv(CSV_DIR / "wave1_3_response_label.csv", dtype=str)
w13_df["pid"] = w13_df["pid"].astype(int)
w13_df = w13_df.set_index("pid")

w4_df = pd.read_csv(CSV_DIR / "wave4_response_label.csv", dtype=str)
w4_df["pid"] = w4_df["pid"].astype(int)
w4_df = w4_df.set_index("pid")

print(f"waves 1-3: {w13_df.shape[0]:,} respondents x {w13_df.shape[1]} columns")
print(f"wave 4:    {w4_df.shape[0]:,} respondents x {w4_df.shape[1]} columns")

if not w13_df.index.sort_values().equals(w4_df.index.sort_values()):
    raise ValueError("Waves 1-3 and wave 4 contain different pids")
missing_in_w13 = sorted(set(w4_df.columns) - set(w13_df.columns))
if missing_in_w13:
    raise ValueError(f"Wave 4 columns without a waves 1-3 counterpart: {missing_in_w13}")

all_cat_cols = [c for cols in catalog_df["csv_columns"] for c in cols]
if len(all_cat_cols) != len(set(all_cat_cols)):
    raise ValueError("Duplicate column names in catalog csv_columns")
if set(all_cat_cols) != set(w13_df.columns):
    raise ValueError("Catalog csv_columns do not match w13_df columns")
print(f"Verified 1-to-1 bijection between catalog csv_columns and w13_df ({len(all_cat_cols)} columns).")

waves 1-3: 2,058 respondents x 760 columns
wave 4:    2,058 respondents x 126 columns
Verified 1-to-1 bijection between catalog csv_columns and w13_df (760 columns).


## 4. Demographics

In [6]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np

DATA_DIR = Path.cwd().parent if Path.cwd().parent.name == "data" else Path.cwd().parent / "data"
CODE_DIR = DATA_DIR.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from data import persona

demo_df = catalog_df.loc[catalog_df["BlockName"] == "Demographics", ["QuestionID", "QuestionText", "Options", "csv_columns"]]
demo_by_qid = demo_df.set_index("QuestionID")

# Load respondent-level ISSP US personas (2015-2024) and SubPOP evaluation groups for side-by-side comparison
issp_resp_frames = []
for p in sorted(DATA_DIR.glob("issp*/issp*_US_long.parquet")):
    df_year = pd.read_parquet(p).drop_duplicates(subset=["CASEID"]).copy()
    df_year["year"] = p.parent.name
    issp_resp_frames.append(df_year)
issp_resp_df = pd.concat(issp_resp_frames, ignore_index=True)

subpop_groups = {}
with (DATA_DIR / "subpop/subpop_eval.jsonl").open() as f:
    for line in f:
        b = json.loads(line)
        for k, v in b["persona"].items():
            if k != "Country" and v not in subpop_groups.setdefault(k, []):
                subpop_groups[k].append(v)

print(f"Twin-2K-500 respondents: {len(w13_df):,} | ISSP US respondents (2015-2024): {len(issp_resp_df):,}")
display(demo_df)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Twin-2K-500 respondents: 2,058 | ISSP US respondents (2015-2024): 14,356


,QuestionID,QuestionText,Options,csv_columns
0,QID11,Which part of the United States do you current...,"[Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME...",[QID11]
1,QID12,What is the sex that you were assigned at birth?,"[Male, Female]",[QID12]
2,QID13,How old are you?,"[18-29, 30-49, 50-64, 65+]",[QID13]
3,QID14,What is the highest level of schooling or degr...,"[Less than high school, High school graduate, ...",[QID14]
4,QID15,What is your race or origin?,"[White, Black, Asian, Hispanic, Other]",[QID15]
5,QID16,Are you a citizen of the United States?,"[Yes, No]",[QID16]
6,QID17,Which of these best describes you?,"[Married, Living with a partner, Divorced, Sep...",[QID17]
7,QID18,"What is your present religion, if any?","[Protestant, Roman Catholic, Mormon, Orthodox,...",[QID18]
8,QID19,"Aside from weddings and funerals, how often do...","[More than once a week, Once a week, Once or t...",[QID19]
9,QID20,"In politics today, do you consider yourself a","[Republican, Democrat, Independent, Something ...",[QID20]


## Inspect personas

In [7]:
# 4.1 SEX (QID12) and AGE (QID13) — mapped via code/data/persona.py
print(f"=== SEX ({persona.PERSONA_NAMES['SEX']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['SEX']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID12"], "SEX", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["SEX"], "SEX", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["SEX"] + [np.nan]))

print(f"=== AGE ({persona.PERSONA_NAMES['AGE']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['AGE']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID13"], "AGE", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["AGE"], "AGE", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["AGE"] + [np.nan]))

=== SEX (Sex) | Global levels: ['Male', 'Female'] ===


,Twin-2K-500,ISSP (2015-2024)
SEX,,
Male,1014.0,6425
Female,1044.0,7901
NaN,NaN,30


=== AGE (Age) | Global levels: ['18-29', '30-49', '50-64', '65+'] ===


,Twin-2K-500,ISSP (2015-2024)
AGE,,
18-29,388.0,2030
30-49,735.0,4810
50-64,658.0,3580
65+,277.0,3520
NaN,NaN,416


In [8]:
# 4.2 US_REG (QID11) — mapped via code/data/persona.py
print(f"=== US_REG ({persona.PERSONA_NAMES['US_REG']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['US_REG']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID11"], "US_REG", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["US_REG"], "US_REG", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["US_REG"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["US_REG"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["US_REG"]))

=== US_REG (Region) | Global levels: ['Northeast', 'Midwest', 'South', 'West'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Northeast,342,2264,True
Midwest,372,3405,False
South,834,5534,True
West,510,3153,False


In [11]:
# 4.3 EDULEVEL (QID14) — mapped via code/data/persona.py
print(f"=== EDULEVEL ({persona.PERSONA_NAMES['EDULEVEL']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['EDULEVEL']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID14"], "EDULEVEL", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["EDULEVEL"], "EDULEVEL", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["EDULEVEL"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["EDULEVEL"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["EDULEVEL"] + [np.nan]))

=== EDULEVEL (Education) | Global levels: ['Less than high school', 'High school or some college', "Associate's degree", 'College graduate/some postgrad'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Less than high school,17.0,1384,True
High school or some college,740.0,6456,False
Associate's degree,253.0,1250,False
College graduate/some postgrad,1048.0,5224,True
NaN,NaN,42,NaN


In [16]:
# 4.4 RACE (QID15) and US_ETHN1 (ISSP Ancestry) — kept separate in code/data/persona.py
print(f"=== RACE ({persona.PERSONA_NAMES['RACE']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['RACE']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID15"], "RACE", "twin2k").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["RACE"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["RACE"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["RACE"]))

print(f"=== US_ETHN1 ({persona.PERSONA_NAMES['US_ETHN1']}) in ISSP (top 10 of {issp_resp_df['US_ETHN1'].nunique()} ancestry labels) ===")
display(issp_resp_df["US_ETHN1"].value_counts(dropna=False).head(10).to_frame("ISSP (2015-2024) n"))

=== RACE (Race) | Global levels: ['White', 'Black', 'Asian', 'Hispanic', 'Other'] ===


,Twin-2K-500,in_SubPOP_eval
White,1361,True
Black,251,True
Asian,140,True
Hispanic,194,True
Other,112,False


=== US_ETHN1 (Ancestry) in ISSP (top 10 of 96 ancestry labels) ===


,ISSP (2015-2024) n
US_ETHN1,
Germany,1923
England and Wales,1916
NaN,1815
Ireland,1220
Mexico,806
Other African,647
Italy,592
American Indian,503
Scotland,364


In [20]:
# 4.5 QID16 (US citizenship — excluded from persona) and MARITAL (QID17) — mapped via code/data/persona.py
print(f"=== MARITAL ({persona.PERSONA_NAMES['MARITAL']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['MARITAL']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID17"], "MARITAL", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["MARITAL"], "MARITAL", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["MARITAL"] + [np.nan]))

=== MARITAL (Marital status) | Global levels: ['Married', 'Living with a partner', 'Divorced', 'Separated', 'Widowed', 'Never been married'] ===


,Twin-2K-500,ISSP (2015-2024)
MARITAL,,
Married,813.0,6281.0
Living with a partner,212.0,NaN
Divorced,218.0,2389.0
Separated,31.0,422.0
Widowed,70.0,1137.0
Never been married,714.0,4090.0
NaN,NaN,37.0


In [24]:
# 4.6 RELIGGRP (QID18) — mapped via code/data/persona.py
print(f"=== RELIGGRP ({persona.PERSONA_NAMES['RELIGGRP']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['RELIGGRP']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID18"], "RELIGGRP", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["RELIGGRP"], "RELIGGRP", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["RELIGGRP"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["RELIGGRP"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["RELIGGRP"] + [np.nan]))

=== RELIGGRP (Religion) | Global levels: ['Protestant', 'Catholic', 'Orthodox', 'Other Christian', 'Jewish', 'Muslim', 'Buddhist', 'Hindu', 'Atheist', 'Agnostic', 'Nothing in particular', 'Other'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Protestant,510.0,6276.0,True
Catholic,358.0,3033.0,False
Orthodox,17.0,87.0,False
Other Christian,15.0,298.0,False
Jewish,39.0,250.0,True
Muslim,18.0,107.0,True
Buddhist,25.0,123.0,False
Hindu,7.0,96.0,True
Atheist,216.0,3635.0,True
Agnostic,311.0,NaN,False


In [28]:
# 4.7 ATTEND (QID19) — mapped via code/data/persona.py
print(f"=== ATTEND ({persona.PERSONA_NAMES['ATTEND']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['ATTEND']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID19"], "ATTEND", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["ATTEND"], "ATTEND", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["ATTEND"] + [np.nan]))

=== ATTEND (Attendance of religious services) | Global levels: ['More than once a week', 'Once a week', 'Once or twice a month', 'A few times a year', 'Seldom', 'Never'] ===


,Twin-2K-500,ISSP (2015-2024)
ATTEND,,
More than once a week,87.0,741
Once a week,295.0,2132
Once or twice a month,129.0,2440
A few times a year,246.0,3255
Seldom,463.0,1429
Never,838.0,4261
NaN,NaN,98


In [30]:
# 4.8 US_PRTY (QID20) — union of levels in code/data/persona.py
print(f"=== US_PRTY ({persona.PERSONA_NAMES['US_PRTY']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['US_PRTY']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID20"], "US_PRTY", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["US_PRTY"], "US_PRTY", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["US_PRTY"]["subpop"].values() for r in persona.GLOBAL_PERSONA_LEVELS["US_PRTY"]}),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["US_PRTY"] + [np.nan]))

=== US_PRTY (Political party) | Global levels: ['Democrat', 'Republican', 'Independent', 'Something else', 'Other candidate'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Democrat,847.0,5174.0,True
Republican,540.0,3514.0,True
Independent,609.0,NaN,False
Something else,62.0,NaN,False
Other candidate,NaN,427.0,False
NaN,NaN,5241.0,NaN


In [35]:
# 4.9 US_INC (QID21) — mapped via code/data/persona.py
inc_levels = ["Less than $30,000", "$30,000 - $50,000", "$50,000 - $75,000", "$75,000 - $89,999", "$75,000 - $100,000", "$90,000 - $109,999", "$100,000 or more"]
print(f"=== US_INC ({persona.PERSONA_NAMES['US_INC']}) | Global levels (ISSP/Twin/SubPOP): {inc_levels} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID21"], "US_INC", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["US_INC"], "US_INC", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["US_INC"]["subpop"].values() for r in inc_levels}),
}).reindex(inc_levels + [np.nan]))

=== US_INC (Income) | Global levels (ISSP/Twin/SubPOP): ['Less than $30,000', '$30,000 - $50,000', '$50,000 - $75,000', '$75,000 - $89,999', '$75,000 - $100,000', '$90,000 - $109,999', '$100,000 or more'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
"Less than $30,000",367.0,3442.0,True
"$30,000 - $50,000",412.0,2119.0,False
"$50,000 - $75,000",411.0,2302.0,False
"$75,000 - $89,999",NaN,1051.0,False
"$75,000 - $100,000",316.0,NaN,False
"$90,000 - $109,999",NaN,1027.0,False
"$100,000 or more",552.0,2968.0,True
NaN,NaN,1447.0,NaN


In [39]:
# 4.10 PARTY_LR (QID22) — union of levels in code/data/persona.py
party_lr_levels = ["Very liberal", "Liberal", "Moderate", "Conservative", "Very conservative", "Other"]
print(f"=== PARTY_LR ({persona.PERSONA_NAMES['PARTY_LR']}) | Global levels (ISSP/Twin/SubPOP): {party_lr_levels} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID22"], "PARTY_LR", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["PARTY_LR"], "PARTY_LR", "issp").value_counts(dropna=False),
    "in_SubPOP_eval": pd.Series({r: r in persona.PERSONA_MAPS["PARTY_LR"]["subpop"].values() for r in party_lr_levels}),
}).reindex(party_lr_levels + [np.nan]))

=== PARTY_LR (Political ideology) | Global levels (ISSP/Twin/SubPOP): ['Very liberal', 'Liberal', 'Moderate', 'Conservative', 'Very conservative', 'Other'] ===


,Twin-2K-500,ISSP (2015-2024),in_SubPOP_eval
Very liberal,345.0,NaN,False
Liberal,564.0,5193.0,True
Moderate,582.0,NaN,True
Conservative,430.0,3514.0,True
Very conservative,137.0,NaN,False
Other,NaN,408.0,False
NaN,NaN,5241.0,NaN


In [47]:
# 4.11 HHSIZE (QID23) — mapped via code/data/persona.py
print(f"=== HHSIZE ({persona.PERSONA_NAMES['HHSIZE']!r}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['HHSIZE']} ===")
hhsize_mapped = persona.map_persona_series(w13_df["QID23"], "HHSIZE", "twin2k")
display(pd.DataFrame({"Twin-2K-500": hhsize_mapped.value_counts(dropna=False)}).reindex(persona.GLOBAL_PERSONA_LEVELS["HHSIZE"]))
print("Example in state['persona']:")
print(json.dumps({"persona": {"Country": "United States", persona.PERSONA_NAMES["HHSIZE"]: hhsize_mapped.iloc[0]}}, indent=2))

=== HHSIZE ('How many people, including yourself, live in your household?') | Global levels: ['1', '2', '3', '4', 'More than 4'] ===


,Twin-2K-500
HHSIZE,
1,412
2,650
3,423
4,352
More than 4,221


Example in state['persona']:
{
  "persona": {
    "Country": "United States",
    "How many people, including yourself, live in your household?": "3"
  }
}


In [50]:
# 4.12 MAINSTAT (QID24) — union of levels in code/data/persona.py
print(f"=== MAINSTAT ({persona.PERSONA_NAMES['MAINSTAT']}) | Global levels: {persona.GLOBAL_PERSONA_LEVELS['MAINSTAT']} ===")
display(pd.DataFrame({
    "Twin-2K-500": persona.map_persona_series(w13_df["QID24"], "MAINSTAT", "twin2k").value_counts(dropna=False),
    "ISSP (2015-2024)": persona.map_persona_series(issp_resp_df["MAINSTAT"], "MAINSTAT", "issp").value_counts(dropna=False),
}).reindex(persona.GLOBAL_PERSONA_LEVELS["MAINSTAT"] + [np.nan]))

=== MAINSTAT (Main activity) | Global levels: ['Full-time employment', 'Part-time employment', 'Self-employed', 'In paid work', 'Unemployed', 'Home-maker', 'Student', 'Retired', 'Permanently sick or disabled', 'Other'] ===


,Twin-2K-500,ISSP (2015-2024)
MAINSTAT,,
Full-time employment,871.0,NaN
Part-time employment,269.0,NaN
Self-employed,280.0,NaN
In paid work,NaN,8285.0
Unemployed,249.0,698.0
Home-maker,66.0,1265.0
Student,78.0,389.0
Retired,245.0,3212.0
Permanently sick or disabled,NaN,119.0


## 5. Substantive questions in Kev format (easy to difficult)

In [8]:
issp_mapped = persona.apply_persona_mapping(issp_resp_df.drop(columns=["year"]), dataset="issp")
twin_mapped = persona.apply_persona_mapping(w13_df, dataset="twin2k")
print(f"Verified apply_persona_mapping on ISSP ({len(issp_mapped):,} respondents x {len(persona.DATASET_PERSONA_COLS['issp'])} attrs) and Twin-2K-500 ({len(twin_mapped):,} respondents x {len(persona.DATASET_PERSONA_COLS['twin2k'])} attrs).")
display(twin_mapped[persona.DATASET_PERSONA_COLS["twin2k"]].head(3))

Verified apply_persona_mapping on ISSP (14,356 respondents x 16 attrs) and Twin-2K-500 (2,058 respondents x 14 attrs).


,c_alphan,US_REG,SEX,AGE,EDULEVEL,RACE,MARITAL,RELIGGRP,ATTEND,US_PRTY,US_INC,PARTY_LR,HHSIZE,MAINSTAT
pid,,,,,,,,,,,,,,
1,United States,Midwest,Female,30-49,High school or some college,Black,Married,Nothing in particular,A few times a year,Independent,"$50,000 - $75,000",Moderate,3,Part-time employment
2,United States,Midwest,Female,50-64,High school or some college,White,Divorced,Nothing in particular,Never,Democrat,"$30,000 - $50,000",Very liberal,2,Retired
3,United States,South,Male,65+,High school or some college,White,Married,Protestant,Seldom,Independent,"$30,000 - $50,000",Moderate,2,Retired


In [30]:
# 5.0 Load question_audit.csv, question_overrides.json, long-format export builder, and Kev preview helper
from models.kev_pointer import DEFAULT_BASE, KevEncoder

audit_raw_df = pd.read_csv("question_audit.csv", keep_default_na=False)
overrides_path = Path("question_overrides.json")
if overrides_path.exists():
    question_overrides = json.loads(overrides_path.read_text(encoding="utf-8"))
    if not isinstance(question_overrides, dict):
        raise TypeError(f"{overrides_path.name} must contain a JSON object, got {type(question_overrides).__name__}")
else:
    question_overrides = {}

ALLOWED_OVERRIDE_FIELDS = {"stem", "subitems", "options"}
TERMINAL_PUNCT = frozenset(".?!:;…\"')")


def normalize_multiline_override(text: str) -> str:
    lines = [ln.rstrip() for ln in text.replace("\r\n", "\n").replace("\r", "\n").split("\n")]
    return "\n".join(lines).strip()


valid_override_keys = set()
audit_rows = []
for _, r in audit_raw_df.iterrows():
    idx_str = str(r["catalog_idx"])
    row_key = f"{idx_str}:{r['QuestionID']}"
    valid_override_keys.add(row_key)
    drop_opts = r["drop_options"] if "drop_options" in audit_raw_df.columns else ""
    if ".." in idx_str:
        start_idx, end_idx = map(int, idx_str.split(".."))
        for idx in range(start_idx, end_idx + 1):
            audit_rows.append({
                "catalog_idx": idx,
                "override_key": row_key,
                "QuestionID": f"QID9_{idx - start_idx + 1}",
                "family": r["family"],
                "include": r["include"] == "True" if isinstance(r["include"], str) else bool(r["include"]),
                "kev_type": r["kev_type"],
                "drop_options": drop_opts,
            })
    else:
        audit_rows.append({
            "catalog_idx": int(idx_str),
            "override_key": row_key,
            "QuestionID": r["QuestionID"],
            "family": r["family"],
            "include": r["include"] == "True" if isinstance(r["include"], str) else bool(r["include"]),
            "kev_type": r["kev_type"],
            "drop_options": drop_opts,
        })

tags_path = Path("question_tags.json")
if tags_path.exists():
    question_tags = json.loads(tags_path.read_text(encoding="utf-8"))
    if not isinstance(question_tags, dict):
        raise TypeError(f"{tags_path.name} must contain a JSON object, got {type(question_tags).__name__}")
    for rk_tag, tag_list in question_tags.items():
        if not isinstance(tag_list, list) or not all(isinstance(t, str) and t.strip() for t in tag_list):
            raise ValueError(f"{rk_tag}: question_tags entry must be a list of non-empty strings")
else:
    question_tags = {}

unknown_override_keys = sorted(set(question_overrides.keys()) - valid_override_keys)
if unknown_override_keys:
    raise KeyError(f"Unknown row keys in {overrides_path.name}: {unknown_override_keys}")
unknown_tag_keys = sorted(set(question_tags.keys()) - valid_override_keys)
if unknown_tag_keys:
    raise KeyError(f"Unknown row keys in {tags_path.name}: {unknown_tag_keys}")

audit_expanded_df = pd.DataFrame(audit_rows).set_index("catalog_idx").sort_index()
if not audit_expanded_df.index.equals(catalog_df.index):
    raise ValueError("Expanded question_audit.csv indices do not match catalog_df.index")
if not (audit_expanded_df["QuestionID"] == catalog_df["QuestionID"]).all():
    raise ValueError("Expanded question_audit.csv QuestionIDs do not match catalog_df['QuestionID']")

for col in ("override_key", "family", "include", "kev_type", "drop_options"):
    catalog_df[col] = audit_expanded_df[col]

for rk, ov in question_overrides.items():
    if not isinstance(ov, dict) or not ov:
        raise ValueError(f"{rk}: override entry must be a non-empty dict")
    extra_fields = set(ov.keys()) - ALLOWED_OVERRIDE_FIELDS
    if extra_fields:
        raise ValueError(f"{rk}: unknown override fields {sorted(extra_fields)}")
    matched_cat = catalog_df[catalog_df["override_key"] == rk]
    if matched_cat.empty:
        raise KeyError(f"{rk}: override key did not match any catalog_df row")
    if ".." in rk.split(":", 1)[0] and "stem" in ov:
        raise ValueError(f"{rk}: static stem override is not allowed on collapsed loop rows")
    cat0 = matched_cat.iloc[0]
    if "stem" in ov and not isinstance(ov["stem"], str):
        raise TypeError(f"{rk}: stem override must be str")
    if "subitems" in ov:
        if not isinstance(ov["subitems"], dict) or not ov["subitems"]:
            raise ValueError(f"{rk}: subitems override must be a non-empty dict")
        cat_rows = cat0["Rows"] if isinstance(cat0["Rows"], list) else []
        if not cat_rows:
            raise ValueError(f"{rk}: subitems override set on row without Matrix Rows")
        for sub_k, sub_v in ov["subitems"].items():
            if not isinstance(sub_k, str) or not sub_k.isdigit():
                raise ValueError(f"{rk}: subitem key {sub_k!r} must be a 0-based integer string")
            sub_idx = int(sub_k)
            if not (0 <= sub_idx < len(cat_rows)):
                raise IndexError(f"{rk}: subitem index {sub_idx} out of range (0..{len(cat_rows) - 1})")
            if not isinstance(sub_v, str) or not normalize_multiline_override(sub_v):
                raise ValueError(f"{rk}: subitem {sub_k!r} override must be a non-empty string")
    if "options" in ov:
        if not isinstance(ov["options"], dict) or not ov["options"]:
            raise ValueError(f"{rk}: options override must be a non-empty dict")
        if cat0["family"] == "1_Matrix_Bipolar":
            raise ValueError(f"{rk}: override Bipolar pair options via subitems ('opt1:opt2') rather than options")
        raw_opt_list = cat0["Columns"] if isinstance(cat0["Columns"], list) else (cat0["Options"] if isinstance(cat0["Options"], list) else [])
        known_opts = {" ".join(str(x).split()) for x in raw_opt_list}
        for opt_k, opt_v in ov["options"].items():
            if opt_k not in known_opts:
                raise KeyError(f"{rk}: option override key {opt_k!r} not in {sorted(known_opts)}")
            if not isinstance(opt_v, str) or not " ".join(opt_v.split()):
                raise ValueError(f"{rk}: option override for {opt_k!r} must be a non-empty string")

print(f"Verified 1-to-1 alignment between question_audit.csv, catalog_df (256 catalog entries), and {overrides_path.name} ({len(question_overrides)} overrides):")
display(
    catalog_df.groupby(["family", "include"], as_index=False)
    .agg(n_catalog_entries=("QuestionID", "count"), n_csv_cols=("n_csv_cols", "sum"))
)

kev_encoder = KevEncoder.for_base(DEFAULT_BASE)
PERSONA_COLS = persona.DATASET_PERSONA_COLS["twin2k"]

persona_base_df = twin_mapped[PERSONA_COLS].copy()
persona_base_df.insert(0, "CASEID", [f"twin_{pid}" for pid in persona_base_df.index])


def build_long_df(resp_df, question_specs):
    """Build long-format DataFrame matching the ISSP parquet schema from question_specs."""
    frames = []
    for spec in question_specs:
        col = spec["v"]
        s = resp_df[col].dropna()
        drop_raw = spec["drop_raw"] if "drop_raw" in spec else set()
        if drop_raw:
            s = s[~s.isin(drop_raw)]
        raw_to_code = spec["raw_to_code"]
        unexpected = set(s.unique()) - set(raw_to_code.keys())
        if unexpected:
            raise ValueError(f"{col}: unexpected response values {unexpected}")

        ans_codes = [raw_to_code[val] for val in s]
        code_to_text = dict(zip(spec["option_codes"], spec["option_texts"], strict=True))
        ans_texts = [code_to_text[c] for c in ans_codes]

        sub = persona_base_df.loc[s.index].copy()
        n = len(sub)
        sub["v"] = col
        sub["label"] = spec["label"]
        sub["question"] = spec["question"]
        sub["type"] = spec["type"]
        sub["reversed"] = False
        sub["option_codes"] = [list(spec["option_codes"])] * n
        sub["option_texts"] = [list(spec["option_texts"])] * n
        sub["answer_code"] = ans_codes
        sub["answer_text"] = ans_texts
        frames.append(sub.reset_index(drop=True))
    return pd.concat(frames, ignore_index=True)


def preview_long_question(long_df, v_id, attrs=None, case_id="twin_1", width=90):
    sub_q = long_df[long_df["v"] == v_id]
    ref_row = sub_q[sub_q["CASEID"] == case_id].iloc[0]
    q_type = ref_row["type"]
    opt_codes = ref_row["option_codes"]
    opt_texts = ref_row["option_texts"]

    if attrs is None:
        cols = [c for c in PERSONA_COLS if c != "c_alphan" and pd.notna(ref_row[c])]
        probs = [1.0 if c == ref_row["answer_code"] else 0.0 for c in opt_codes]
        header_tag = f"CASEID={case_id!r}"
    else:
        cols = list(attrs)
        mask = pd.Series(True, index=sub_q.index)
        for c in cols:
            mask &= sub_q[c] == ref_row[c]
        sub_grp = sub_q.loc[mask]
        counts = sub_grp["answer_code"].value_counts().reindex(opt_codes, fill_value=0)
        probs = (counts / counts.sum()).astype(float).tolist()
        header_tag = f"subpop n={len(sub_grp)}"

    state_lines = [f"{persona.PERSONA_NAMES['c_alphan']}: {ref_row['c_alphan']}"]
    state_lines.extend(f"{persona.PERSONA_NAMES[c]}: {ref_row[c]}" for c in cols)
    if q_type == "score":
        criteria = list(opt_texts)
        target = {str(i): round(p, 4) for i, p in enumerate(probs)}
    elif q_type == "choice":
        criteria = {t: None for t in opt_texts}
        target = {t: round(p, 4) for t, p in zip(opt_texts, probs)}
    else:
        raise ValueError(f"Unexpected type {q_type!r}")

    view = {
        "state": "\n".join(state_lines),
        "questions": {
            v_id: {
                "type": q_type,
                "instructions": ref_row["question"],
                "criteria": criteria,
                "target": target,
            }
        },
    }
    header, rendered = kev_encoder.preview(view)
    wrapped = "\n".join(
        line if line.startswith("<|") else textwrap.fill(line, width=width)
        for line in rendered.splitlines()
    )
    print(f"\n=== {v_id} (type={q_type!r}, {header_tag}, {header}) ===")
    print(wrapped)
    print("Target distribution:", json.dumps(target))

Verified 1-to-1 alignment between question_audit.csv and catalog_df (256 catalog entries):


,family,include,n_catalog_entries,n_csv_cols
0,0_DB,False,14,0
1,0_Demographics,False,14,14
2,0_Slider,False,3,12
3,0_TE,False,28,62
4,1_Matrix_Bipolar,True,13,158
5,2_Matrix_Likert,True,23,290
6,3_MC_Pricing,True,40,40
7,4_MC_Single,False,14,14
8,4_MC_Single,True,86,86
9,5_MC_Multi,False,21,84


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [37]:
catalog_df[catalog_df['include']==False]

,QuestionID,QuestionText,QuestionType,Options,Settings,BlockName,source,csv_columns,is_descriptive,Rows,Columns,RowsID,Statements,Range,StatementsID,Selector,n_csv_cols,family,include,kev_type
0,QID11,Which part of the United States do you current...,MC,"[Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME...","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID11],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1,0_Demographics,False,
1,QID12,What is the sex that you were assigned at birth?,MC,"[Male, Female]","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID12],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1,0_Demographics,False,
2,QID13,How old are you?,MC,"[18-29, 30-49, 50-64, 65+]","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID13],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1,0_Demographics,False,
3,QID14,What is the highest level of schooling or degr...,MC,"[Less than high school, High school graduate, ...","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID14],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1,0_Demographics,False,
4,QID15,What is your race or origin?,MC,"[White, Black, Asian, Hispanic, Other]","{'Selector': 'SAVR', 'SubSelector': 'TX', 'For...",Demographics,wave1_3_persona_json,[QID15],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SAVR,1,0_Demographics,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,QID8,You will now be asked a series of 40 questions...,DB,NaN,NaN,Product Preferences - Pricing,wave4_Q_wave1_3_A,[],True,NaN,NaN,NaN,NaN,NaN,NaN,,0,0_DB,False,
237,QID164,How many African countries do you think are in...,TE,NaN,"{'Selector': 'SL', 'ForceResponse': 'ON', 'Con...",Anchoring - African countries low,wave4_Q_wave1_3_A,[QID164_TEXT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SL,1,0_TE,False,
239,QID168,How tall do you think the tallest redwood tree...,TE,NaN,"{'Selector': 'SL', 'ForceResponse': 'ON', 'Con...",Anchoring - redwood low,wave4_Q_wave1_3_A,[QID168_TEXT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SL,1,0_TE,False,
240,QID182,Imagine that you just paid $50 for a Coffee Co...,TE,NaN,"{'Selector': 'SL', 'ForceResponse': 'ON', 'Con...",Sunk cost - yes,wave4_Q_wave1_3_A,[QID182_TEXT],NaN,NaN,NaN,NaN,NaN,NaN,NaN,SL,1,0_TE,False,


In [38]:
# 5.1 Family 1: Matrix — Bipolar export pipeline (filtered by question_audit.csv + question_overrides.json)
bipolar_entries = catalog_df[(catalog_df["family"] == "1_Matrix_Bipolar") & catalog_df["include"]]
bipolar_specs = []
for _, q in bipolar_entries.iterrows():
    ov = question_overrides[q["override_key"]] if q["override_key"] in question_overrides else None
    q_text = normalize_multiline_override(ov["stem"]) if ov is not None and "stem" in ov else " ".join(q["QuestionText"].split())
    q_type = q["kev_type"]
    if q_type != "choice":
        raise ValueError(f"{q['QuestionID']}: expected kev_type='choice', got {q_type!r}")
    rows = [
        normalize_multiline_override(ov["subitems"][str(i)]) if ov is not None and "subitems" in ov and str(i) in ov["subitems"] else r_str
        for i, r_str in enumerate(q["Rows"])
    ]
    for col, row_str in zip(q["csv_columns"], rows, strict=True):
        opt1, opt2 = row_str.split(":")
        bipolar_specs.append({
            "v": col,
            "label": f"{q['BlockName'].strip()}: {row_str}",
            "question": q_text,
            "type": q_type,
            "option_codes": [1, 2],
            "option_texts": [opt1, opt2],
            "raw_to_code": {"1": 1, "2": 2},
        })

bipolar_long_df = build_long_df(w13_df, bipolar_specs)
print(f"bipolar_long_df: {len(bipolar_long_df):,} rows x {bipolar_long_df.shape[1]} cols ({bipolar_long_df['v'].nunique()} questions, {bipolar_long_df['CASEID'].nunique():,} respondents)")
display(bipolar_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_codes", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["QID84_4", "QID250_1", "QID279_1"]:
    preview_long_question(bipolar_long_df, v_id)

bipolar_long_df: 325,164 rows x 24 cols (158 questions, 2,058 respondents)


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_codes,option_texts,answer_code,answer_text
0,twin_1,Female,30-49,High school or some college,QID84_4,Economic preferences: $6.00 in 6 weeks:$3.00 i...,choice,"[1, 2]","[$6.00 in 6 weeks, $3.00 in 5 weeks]",1,$6.00 in 6 weeks
1,twin_2,Female,50-64,High school or some college,QID84_4,Economic preferences: $6.00 in 6 weeks:$3.00 i...,choice,"[1, 2]","[$6.00 in 6 weeks, $3.00 in 5 weeks]",1,$6.00 in 6 weeks
2,twin_3,Male,65+,High school or some college,QID84_4,Economic preferences: $6.00 in 6 weeks:$3.00 i...,choice,"[1, 2]","[$6.00 in 6 weeks, $3.00 in 5 weeks]",1,$6.00 in 6 weeks
3,twin_4,Male,50-64,College graduate/some postgrad,QID84_4,Economic preferences: $6.00 in 6 weeks:$3.00 i...,choice,"[1, 2]","[$6.00 in 6 weeks, $3.00 in 5 weeks]",1,$6.00 in 6 weeks



=== QID84_4 (type='choice', CASEID='twin_1', 210 tokens; state 108 of at most 384) ===
<|fim_prefix|>
Country: United States
Region: Midwest
Sex: Female
Age: 30-49
Education: High school or some college
Race: Black
Marital status: Married
Religion: Nothing in particular
Attendance of religious services: A few times a year
Political party: Independent
Income: $50,000 - $75,000
Political ideology: Moderate
How many people, including yourself, live in your household?: 3
Main activity: Part-time employment
<|fim_middle|>
Please choose between the following options. For each line in the list, you must choose
between the option on the left and the option on the right. Note that on each line, the
option on the left stays the same while the option on the right gets better as one goes
down the list. You can select the option you would prefer receiving by clicking on the
button next to that option.
<|box_start|>$6.00 in 6 weeks<|box_end|>
<|box_start|>$3.00 in 5 weeks<|box_end|>
<|fim_suffix|>


In [39]:
# 5.2 Family 2: Matrix — Likert export pipeline (filtered by question_audit.csv + question_overrides.json)
likert_entries = catalog_df[(catalog_df["family"] == "2_Matrix_Likert") & catalog_df["include"]]
likert_specs = []
for _, q in likert_entries.iterrows():
    ov = question_overrides[q["override_key"]] if q["override_key"] in question_overrides else None
    q_text = normalize_multiline_override(ov["stem"]) if ov is not None and "stem" in ov else " ".join(q["QuestionText"].split())
    drop_set = {x.strip() for x in q["drop_options"].split("|") if x.strip()}
    raw_cols = list(q["Columns"])
    drop_raw = {raw_c for raw_c in raw_cols if " ".join(raw_c.split()) in drop_set}
    kept_raw_cols = [raw_c for raw_c in raw_cols if raw_c not in drop_raw]
    opt_ov = ov["options"] if ov is not None and "options" in ov else {}
    opts = [" ".join(opt_ov[" ".join(raw_c.split())].split()) if " ".join(raw_c.split()) in opt_ov else raw_c for raw_c in kept_raw_cols]
    if len(opts) < 2 or len(set(opts)) != len(opts):
        raise ValueError(f"{q['QuestionID']}: invalid Likert options after override: {opts}")
    codes = list(range(1, len(opts) + 1))
    q_type = q["kev_type"]
    if q_type not in ("score", "choice"):
        raise ValueError(f"{q['QuestionID']}: unexpected kev_type {q_type!r}")
    raw_to_code = {raw_c: c for c, raw_c in zip(codes, kept_raw_cols, strict=True)}
    rows = [
        normalize_multiline_override(ov["subitems"][str(i)]) if ov is not None and "subitems" in ov and str(i) in ov["subitems"] else r_str
        for i, r_str in enumerate(q["Rows"])
    ]
    for col, row_str in zip(q["csv_columns"], rows, strict=True):
        joined_q = f"{q_text} {row_str}" if (not q_text or q_text[-1] in TERMINAL_PUNCT) else f"{q_text}\n\n{row_str}"
        likert_specs.append({
            "v": col,
            "label": f"{q['BlockName'].strip()}: {row_str}",
            "question": joined_q if q_text else row_str,
            "type": q_type,
            "option_codes": codes,
            "option_texts": opts,
            "raw_to_code": raw_to_code,
            "drop_raw": drop_raw,
        })

likert_long_df = build_long_df(w13_df, likert_specs)
print(
    f"likert_long_df: {len(likert_long_df):,} rows x {likert_long_df.shape[1]} cols "
    f"({likert_long_df['v'].nunique()} questions: "
    f"{likert_long_df[likert_long_df['type'] == 'score']['v'].nunique()} 'score', "
    f"{likert_long_df[likert_long_df['type'] == 'choice']['v'].nunique()} 'choice')"
)
display(likert_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_codes", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["QID25_1", "QID287_1", "QID29_1", "QID27_1", "QID203_1"]:
    preview_long_question(likert_long_df, v_id)

likert_long_df: 574,053 rows x 24 cols (290 questions: 261 'score', 29 'choice')


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_codes,option_texts,answer_code,answer_text
0,twin_1,Female,30-49,High school or some college,QID25_1,Personality: Is talkative,score,"[1, 2, 3, 4, 5]","[Disagree strongly, Disagree a little, Neither...",1,Disagree strongly
1,twin_2,Female,50-64,High school or some college,QID25_1,Personality: Is talkative,score,"[1, 2, 3, 4, 5]","[Disagree strongly, Disagree a little, Neither...",4,Agree a little
2,twin_3,Male,65+,High school or some college,QID25_1,Personality: Is talkative,score,"[1, 2, 3, 4, 5]","[Disagree strongly, Disagree a little, Neither...",2,Disagree a little
3,twin_4,Male,50-64,College graduate/some postgrad,QID25_1,Personality: Is talkative,score,"[1, 2, 3, 4, 5]","[Disagree strongly, Disagree a little, Neither...",5,Agree strongly



=== QID25_1 (type='score', CASEID='twin_1', 181 tokens; state 108 of at most 384) ===
<|fim_prefix|>
Country: United States
Region: Midwest
Sex: Female
Age: 30-49
Education: High school or some college
Race: Black
Marital status: Married
Religion: Nothing in particular
Attendance of religious services: A few times a year
Political party: Independent
Income: $50,000 - $75,000
Political ideology: Moderate
How many people, including yourself, live in your household?: 3
Main activity: Part-time employment
<|fim_middle|>
Here are a number of characteristics that may or may not apply to you. Please indicate
next to each statement the extent to which you agree or disagree with that statement. I
see myself as someone who... Is talkative
<|box_start|>Disagree strongly<|box_end|>
<|box_start|>Disagree a little<|box_end|>
<|box_start|>Neither agree nor disagree<|box_end|>
<|box_start|>Agree a little<|box_end|>
<|box_start|>Agree strongly<|box_end|>
<|fim_suffix|>
Target distribution: {"0": 1.0, 

In [40]:
# 5.3 Family 3: MC single-select — Product Preferences - Pricing export pipeline (filtered by question_audit.csv + question_overrides.json)
pricing_entries = catalog_df[(catalog_df["family"] == "3_MC_Pricing") & catalog_df["include"]]
if len(pricing_entries) != 40:
    raise ValueError(f"Expected 40 included pricing entries in question_audit.csv, got {len(pricing_entries)}")
pricing_kev_types = set(pricing_entries["kev_type"].unique())
if pricing_kev_types != {"choice"}:
    raise ValueError(f"Expected pricing kev_type='choice', got {pricing_kev_types}")
pricing_ov_key = pricing_entries.iloc[0]["override_key"]
pricing_opt_ov = (
    question_overrides[pricing_ov_key]["options"]
    if pricing_ov_key in question_overrides and "options" in question_overrides[pricing_ov_key]
    else {}
)

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR,
    allow_patterns=["wave_split/*"],
)
wave_split_df = pd.read_parquet(LOCAL_DIR / "wave_split" / "chunks")
wave_split_df["pid"] = wave_split_df["pid"].astype(int)
wave_split_df = wave_split_df.set_index("pid").sort_index()

if not wave_split_df.index.equals(w13_df.index.sort_values()):
    raise ValueError("wave_split_df and w13_df have different pids")

# 1) Verify every respondent's Demographics in wave_split_df matches w13_df (2,058 x 14 = 28,812 cells)
for pid, j_str in wave_split_df["wave1_3_persona_json"].items():
    for b in json.loads(j_str):
        if b["BlockName"] == "Demographics":
            for q in b["Questions"]:
                qid = q["QuestionID"]
                ws_ans = q["Options"][int(q["Answers"]["SelectedByPosition"]) - 1]
                csv_ans = w13_df.loc[pid, qid]
                if ws_ans != csv_ans:
                    raise ValueError(f"Demographics mismatch for pid={pid}, {qid}: {ws_ans!r} != {csv_ans!r}")

# 2) Extract Pricing questions and verify every answer in wave_split_df matches w13_df (2,058 x 40 = 82,320 cells)
pricing_records = []
for pid, j_str in wave_split_df["wave4_Q_wave1_3_A"].items():
    for b in json.loads(j_str):
        if b["BlockName"] == "Product Preferences - Pricing":
            for q in b["Questions"]:
                if q["QuestionType"] == "MC":
                    slot_qid = q["QuestionID"]
                    q_text = " ".join(q["QuestionText"].split())
                    raw_opts = list(q["Options"])
                    ans_code = int(q["Answers"]["SelectedByPosition"])
                    raw_ans_text = raw_opts[ans_code - 1]
                    csv_ans = w13_df.loc[pid, slot_qid]
                    if raw_ans_text != csv_ans:
                        raise ValueError(f"Pricing answer mismatch for pid={pid}, {slot_qid}: {raw_ans_text!r} != {csv_ans!r}")
                    opts = [" ".join(pricing_opt_ov[" ".join(o.split())].split()) if " ".join(o.split()) in pricing_opt_ov else o for o in raw_opts]
                    if len(set(opts)) != len(opts):
                        raise ValueError(f"Pricing duplicate options after override: {opts}")
                    pricing_records.append({
                        "pid": pid,
                        "slot_qid": slot_qid,
                        "question": q_text,
                        "option_texts": opts,
                        "answer_code": ans_code,
                        "answer_text": opts[ans_code - 1],
                    })

raw_pricing_df = pd.DataFrame(pricing_records)
unique_qtexts = sorted(raw_pricing_df["question"].unique())
qtext_to_v = {qt: f"QID9_{i:03d}" for i, qt in enumerate(unique_qtexts, 1)}
raw_pricing_df["v"] = raw_pricing_df["question"].map(qtext_to_v)
raw_pricing_df = raw_pricing_df.sort_values(["v", "pid"], ignore_index=True)
raw_pricing_df["label"] = "Product Preferences - Pricing: " + raw_pricing_df["v"]
raw_pricing_df["type"] = "choice"
raw_pricing_df["reversed"] = False
raw_pricing_df["option_codes"] = [[1, 2]] * len(raw_pricing_df)

pricing_long_df = persona_base_df.loc[raw_pricing_df["pid"]].reset_index(drop=True)
for c in ["v", "label", "question", "type", "reversed", "option_codes", "option_texts", "answer_code", "answer_text"]:
    pricing_long_df[c] = raw_pricing_df[c].values

# 3) Verify post-join alignment: CASEID and answer_text match w13_df[pid, slot_qid] for all 82,320 rows
if (pricing_long_df["CASEID"] != ("twin_" + raw_pricing_df["pid"].astype(str))).any():
    raise ValueError("Post-join CASEID misalignment")
print("Verified 100% match between wave_split_df and w13_df across all 28,812 demographic cells and 82,320 pricing cells.")

q_counts = pricing_long_df.groupby("v")["CASEID"].count()
print(
    f"pricing_long_df: {len(pricing_long_df):,} rows x {pricing_long_df.shape[1]} cols "
    f"({pricing_long_df['v'].nunique()} unique questions, "
    f"n per question: min={q_counts.min()}, median={q_counts.median():.0f}, max={q_counts.max()})"
)
display(pricing_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_codes", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["QID9_152", "QID9_304"]:
    preview_long_question(pricing_long_df, v_id)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Verified 100% match between wave_split_df and w13_df across all 28,812 demographic cells and 82,320 pricing cells.
pricing_long_df: 82,320 rows x 24 cols (440 unique questions, n per question: min=146, median=187, max=226)


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_codes,option_texts,answer_code,answer_text
0,twin_2,Female,50-64,High school or some college,QID9_001,Product Preferences - Pricing: QID9_001,choice,"[1, 2]","[Yes, I would purchase the product, No, I woul...",1,"Yes, I would purchase the product"
1,twin_11,Female,50-64,Associate's degree,QID9_001,Product Preferences - Pricing: QID9_001,choice,"[1, 2]","[Yes, I would purchase the product, No, I woul...",1,"Yes, I would purchase the product"
2,twin_12,Male,50-64,College graduate/some postgrad,QID9_001,Product Preferences - Pricing: QID9_001,choice,"[1, 2]","[Yes, I would purchase the product, No, I woul...",1,"Yes, I would purchase the product"
3,twin_16,Male,50-64,High school or some college,QID9_001,Product Preferences - Pricing: QID9_001,choice,"[1, 2]","[Yes, I would purchase the product, No, I woul...",1,"Yes, I would purchase the product"



=== QID9_152 (type='choice', CASEID='twin_1', 197 tokens; state 108 of at most 384) ===
<|fim_prefix|>
Country: United States
Region: Midwest
Sex: Female
Age: 30-49
Education: High school or some college
Race: Black
Marital status: Married
Religion: Nothing in particular
Attendance of religious services: A few times a year
Political party: Independent
Income: $50,000 - $75,000
Political ideology: Moderate
How many people, including yourself, live in your household?: 3
Main activity: Part-time employment
<|fim_middle|>
Please consider the following product category: Dairy Products. Suppose you are in a
grocery store, and you see the following product in that category: Land O Lakes Salted
Stick Butter, 16 oz, 4 Sticks. The product is priced at: $7.39. Would you or would you not
purchase this product?
<|box_start|>Yes, I would purchase the product<|box_end|>
<|box_start|>No, I would not purchase the product<|box_end|>
<|fim_suffix|>
Target distribution: {"Yes, I would purchase the produc

In [41]:
# 5.4 Family 4: MC single-select — 86 included items (filtered by question_audit.csv + question_overrides.json)
mc_entries = catalog_df[(catalog_df["family"] == "4_MC_Single") & catalog_df["include"]]
mc_excluded = catalog_df[(catalog_df["family"] == "4_MC_Single") & (~catalog_df["include"])]
print(f"4_MC_Single: {len(mc_entries)} included, {len(mc_excluded)} excluded ({list(mc_excluded['QuestionID'])})")

mc_specs = []
for _, q in mc_entries.iterrows():
    if len(q["csv_columns"]) != 1:
        raise ValueError(f"{q['QuestionID']}: expected 1 csv_column, got {q['csv_columns']}")
    col = q["csv_columns"][0]
    block = q["BlockName"].strip()
    ov = question_overrides[q["override_key"]] if q["override_key"] in question_overrides else None
    q_text = normalize_multiline_override(ov["stem"]) if ov is not None and "stem" in ov else " ".join(q["QuestionText"].split())
    drop_set = {x.strip() for x in q["drop_options"].split("|") if x.strip()}
    raw_opts = list(q["Options"])
    drop_raw = {raw_o for raw_o in raw_opts if " ".join(raw_o.split()) in drop_set}
    kept_raw_opts = [raw_o for raw_o in raw_opts if raw_o not in drop_raw]
    opt_ov = ov["options"] if ov is not None and "options" in ov else {}
    opts = [" ".join(opt_ov[" ".join(raw_o.split())].split()) if " ".join(raw_o.split()) in opt_ov else raw_o for raw_o in kept_raw_opts]
    if len(opts) < 2 or len(set(opts)) != len(opts):
        raise ValueError(f"{q['QuestionID']}: invalid MC options after override: {opts}")
    codes = list(range(1, len(opts) + 1))
    q_type = q["kev_type"]
    if q_type not in ("score", "choice"):
        raise ValueError(f"{q['QuestionID']}: unexpected kev_type {q_type!r}")
    raw_to_code = {raw_o: c for c, raw_o in zip(codes, kept_raw_opts, strict=True)}
    mc_specs.append({
        "v": col,
        "label": f"{block}: {col}",
        "question": q_text,
        "type": q_type,
        "option_codes": codes,
        "option_texts": opts,
        "raw_to_code": raw_to_code,
        "drop_raw": drop_raw,
    })

mc_long_df = build_long_df(w13_df, mc_specs)
print(
    f"mc_long_df: {len(mc_long_df):,} rows x {mc_long_df.shape[1]} cols "
    f"({mc_long_df['v'].nunique()} questions: "
    f"{mc_long_df[mc_long_df['type'] == 'choice']['v'].nunique()} 'choice', "
    f"{mc_long_df[mc_long_df['type'] == 'score']['v'].nunique()} 'score')"
)
display(mc_long_df[["CASEID", "SEX", "AGE", "EDULEVEL", "v", "label", "type", "option_codes", "option_texts", "answer_code", "answer_text"]].head(4))

for v_id in ["QID63", "QID31", "QID158", "QID224", "QID117", "QID278"]:
    preview_long_question(mc_long_df, v_id)

4_MC_Single: 86 included, 14 excluded (['QID268', 'QID269', 'QID270', 'QID225', 'QID226', 'QID227', 'QID228', 'QID229', 'QID230', 'QID118', 'QID119', 'QID120', 'QID121', 'QID122'])
mc_long_df: 146,112 rows x 24 cols (86 questions: 60 'choice', 26 'score')


,CASEID,SEX,AGE,EDULEVEL,v,label,type,option_codes,option_texts,answer_code,answer_text
0,twin_1,Female,30-49,High school or some college,QID56,Cognitive tests: QID56,choice,"[1, 2, 3, 4, 5, 6]","[A, B, C, D, E, F]",2,B
1,twin_2,Female,50-64,High school or some college,QID56,Cognitive tests: QID56,choice,"[1, 2, 3, 4, 5, 6]","[A, B, C, D, E, F]",4,D
2,twin_3,Male,65+,High school or some college,QID56,Cognitive tests: QID56,choice,"[1, 2, 3, 4, 5, 6]","[A, B, C, D, E, F]",1,A
3,twin_4,Male,50-64,College graduate/some postgrad,QID56,Cognitive tests: QID56,choice,"[1, 2, 3, 4, 5, 6]","[A, B, C, D, E, F]",4,D



=== QID63 (type='choice', CASEID='twin_1', 150 tokens; state 108 of at most 384) ===
<|fim_prefix|>
Country: United States
Region: Midwest
Sex: Female
Age: 30-49
Education: High school or some college
Race: Black
Marital status: Married
Religion: Nothing in particular
Attendance of religious services: A few times a year
Political party: Independent
Income: $50,000 - $75,000
Political ideology: Moderate
How many people, including yourself, live in your household?: 3
Main activity: Part-time employment
<|fim_middle|>
Synonym: Select the word that is most nearly the same in meaning as CONCUR
<|box_start|>acquiesce<|box_end|>
<|box_start|>extricate<|box_end|>
<|box_start|>divulge<|box_end|>
<|box_start|>concoct<|box_end|>
<|box_start|>ransack<|box_end|>
<|fim_suffix|>
Target distribution: {"acquiesce": 0.0, "extricate": 0.0, "divulge": 1.0, "concoct": 0.0, "ransack": 0.0}

=== QID31 (type='score', CASEID='twin_1', 180 tokens; state 108 of at most 384) ===
<|fim_prefix|>
Country: United St

In [42]:
# 5.5 Combine all included families and export twin2k_US_long.parquet
multi_included = catalog_df[(catalog_df["family"] == "5_MC_Multi") & catalog_df["include"]]
if len(multi_included) != 0:
    raise ValueError(f"Expected 0 included 5_MC_Multi entries, got {len(multi_included)}")

twin2k_long_df = pd.concat(
    [bipolar_long_df, likert_long_df, pricing_long_df, mc_long_df],
    ignore_index=True,
)

out_parquet = Path("twin2k_US_long.parquet")
twin2k_long_df.to_parquet(out_parquet, index=False)

print(
    f"Saved {out_parquet} ({len(twin2k_long_df):,} rows x {twin2k_long_df.shape[1]} cols | "
    f"{twin2k_long_df['CASEID'].nunique():,} respondents | "
    f"{twin2k_long_df['v'].nunique()} unique questions: "
    f"{twin2k_long_df[twin2k_long_df['type'] == 'choice']['v'].nunique()} 'choice', "
    f"{twin2k_long_df[twin2k_long_df['type'] == 'score']['v'].nunique()} 'score')"
)
display(
    pd.DataFrame({
        "family": ["1_Matrix_Bipolar", "2_Matrix_Likert", "3_MC_Pricing", "4_MC_Single", "Total"],
        "n_questions": [
            bipolar_long_df["v"].nunique(),
            likert_long_df["v"].nunique(),
            pricing_long_df["v"].nunique(),
            mc_long_df["v"].nunique(),
            twin2k_long_df["v"].nunique(),
        ],
        "n_rows": [
            len(bipolar_long_df),
            len(likert_long_df),
            len(pricing_long_df),
            len(mc_long_df),
            len(twin2k_long_df),
        ],
    })
)

Saved twin2k_US_long.parquet (1,127,649 rows x 24 cols | 2,058 respondents | 974 unique questions: 687 'choice', 287 'score')


,family,n_questions,n_rows
0,1_Matrix_Bipolar,158,325164
1,2_Matrix_Likert,290,574053
2,3_MC_Pricing,440,82320
3,4_MC_Single,86,146112
4,Total,974,1127649
